# LATeachingSuite demo

`LATeachingSuite` is the Julia-facing umbrella package in this stack. It re-exports exact problem generators from `GenLAProblems`, owns the `ShowGE` workflow surface, and provides Python-backed GE/QR/eigen/SVD rendering helpers at the top level.


In [ ]:
using LATeachingSuite, LAlatex, LinearAlgebra, LaTeXStrings, Random
Random.seed!(42)


## What this notebook covers

- exact system generation through the umbrella package
- the `ShowGE` workflow for consistent, inconsistent, normal-equation, and multi-RHS cases
- top-level QR, eigenvalue, and SVD render helpers
- canonical `*_bundle` helpers that return both an SVG wrapper and the underlying Python-backed spec payload
- bridge/version helpers for the Python rendering stack


## Exact system generation

A first use of `LATeachingSuite` is simply to generate and display exact matrices from the re-exported `GenLAProblems` surface.


In [ ]:
A_ge, X_ge, B_ge = gen_gj_pb(3, 4, 3; maxint=2, num_rhs=1)
l_show("A X = B : ", L"\qquad ", A_ge, X_ge, " = ", B_ge)


## `ShowGE` on a consistent system

The `ShowGE` workflow is the main Julia-facing entrypoint for elimination, layout, and solution displays. A single `ShowGE(A, B)` stores one RHS matrix `B`; use `b_mat` to choose an RHS matrix and `b_col` to choose a column within that matrix.


In [ ]:
pb_ge = ShowGE{Rational{Int}}(A_ge, B_ge)
show_system(pb_ge; b_mat=1, b_col=1)


In [ ]:
ref!(pb_ge; gj=true)
show_layout!(pb_ge; fig_scale=1.05)


In [ ]:
show_backsubstitution!(pb_ge; b_mat=1, b_col=1, fig_scale=1.05)


In [ ]:
show_solution!(pb_ge; b_mat=1, b_col=1, fig_scale=1.05)


In [ ]:
Xp_ge, Xh_ge = solutions(pb_ge; b_mat=1)
l_show("x_p = ", Xp_ge, L"\qquad x_h = ", Xh_ge)


## Inconsistent systems

`LATeachingSuite` can also work with systems whose right-hand sides lie outside the column space of `A`.


In [ ]:
A_bad, B_bad = gen_inconsistent_gj_pb(4, 6, 3; maxint=2, num_rhs=1)
l_show("Inconsistent A x = b : ", L"\qquad ", A_bad, " x = ", B_bad)


In [ ]:
pb_bad = ShowGE{Rational{Int}}(A_bad, B_bad)
ref!(pb_bad; gj=true)
show_layout!(pb_bad; fig_scale=1.05)


## Multiple RHS matrices

`ShowGE(A, (B1, B2, ...))` stores several RHS matrices. Use `rhs_matrix`, `rhs_column`, and `solutions(...; b_mat=..., b_col=...)` to inspect one selected RHS matrix or one selected RHS column.


In [ ]:
A_multi, X_multi, B_multi = gen_gj_pb(3, 4, 3; maxint=2, num_rhs=2)
B_split = (B_multi[:, 1:1], B_multi[:, 2:2])
pb_multi = ShowGE{Rational{Int}}(A_multi, B_split)
ref!(pb_multi; gj=true)
show_layout!(pb_multi; fig_scale=1.05)


In [ ]:
B1_final = rhs_matrix(pb_multi, 1)
b2 = rhs_column(pb_multi, 2, 1)
xp2, xh2 = solutions(pb_multi; b_mat=2, b_col=1)
l_show("rhs_matrix(pb_multi, 1) = ", B1_final, L"\qquad ", "rhs_column(pb_multi, 2, 1) = ", b2, L"\qquad ", "solutions(pb_multi; b_mat=2, b_col=1): x_p = ", xp2, L"\qquad x_h = ", xh2)


## Normal equations / least squares layout

The same workflow object can drive a normal-equation reduction when you want a least-squares style teaching display.


In [ ]:
A_ls, X_ls, B_ls = gen_gj_pb(3, 4, 2; maxint=2, pivot_in_first_col=true, num_rhs=1, has_zeros=true)
pb_ls = ShowGE{Rational{Int}}(A_ls, B_ls)
show_system(pb_ls; b_mat=1, b_col=1)


In [ ]:
ref!(pb_ls; normal_eq=true)
show_layout!(pb_ls; fig_scale=1.05)


## QR rendering helpers

`LATeachingSuite` exposes QR rendering at the top level; you do not need to call into `nM` or `LAFigureSpecs` directly.


In [ ]:
A_qr = gen_qr_problem(3; family=:pythagorean, maxint=2)
qr_svg(A_qr)


In [ ]:
svg_qr, qr_spec = qr_bundle(A_qr)
Q_qr = q_factor_from_spec(qr_spec)
R_qr = r_factor_from_spec(qr_spec)
l_show("Q = ", Q_qr, L"\qquad ", "R = ", R_qr)


## Eigenvalue rendering helpers


In [ ]:
S_eig, Lambda_eig, S_inv_eig, A_eig = gen_eigenproblem([3, -1, 2]; maxint=2)
l_show("A = ", A_eig, L"\qquad \Lambda = ", Lambda_eig)


In [ ]:
eig_svg(A_eig)


In [ ]:
svg_eig, eig_spec = eig_bundle(A_eig)
Λ_eig, V_eig = eig_matrices_from_spec(eig_spec)
l_show("\Lambda = ", Λ_eig, L"\qquad ", "V = ", V_eig)


Use the bundle spec to recover the assembled matrices or query the eigenvectors attached to a particular eigenvalue.


In [ ]:
Λ_eig, V_eig = eig_matrices_from_spec(eig_spec)
eig_eigenvalues(eig_spec), eig_eigenvectors(eig_spec, eig_spec["lambda"][1])


## SVD rendering helpers


In [ ]:
U_svd, Sigma_svd, Vt_svd, A_svd = gen_svd_problem([2, 1], [2, 1], [3, 1, 0]; maxint=2)
l_show("A = ", A_svd, L"\qquad \Sigma = ", Sigma_svd)


In [ ]:
svd_svg(A_svd)


In [ ]:
svg_svd, svd_spec = svd_bundle(A_svd)
U_svd2, Σ_svd2, V_svd2, rank_svd = svd_matrices_from_spec(svd_spec)
σ1 = svd_singular_values(svd_spec)[1][2]
U1 = svd_left_vectors(svd_spec, σ1)
V1 = svd_right_vectors(svd_spec, σ1)
l_show("U = ", U_svd2, L"\qquad ", "\Sigma = ", Σ_svd2, L"\qquad ", "V = ", V_svd2)


SVD bundle specs support both full matrix reconstruction and semantic queries such as rank or the left/right singular vectors attached to a chosen singular value.


In [ ]:
σ1 = svd_singular_values(svd_spec)[1][2]
U1 = svd_left_vectors(svd_spec, σ1)
V1 = svd_right_vectors(svd_spec, σ1)
(svd_singular_values(svd_spec), rank_svd, U1, V1)


## Python bridge access

`LATeachingSuite` also exposes bridge/version helpers for the Python rendering stack.


In [ ]:
(la_version(), ml_version())
